# 42 · Batch processing
Level L4 · Part 5 · professional · ~25 min · builds on 12

## Goal
Summarise 50 of Scout's notes two ways, 50 live calls and one batch job, compare cost (batch is billed
at half price) and wait time, and decide with a small rule when the batch's latency is acceptable.

## The idea
A live call answers in about a second, at full price. Many jobs do not need that: re-summarising every
note overnight, scoring an eval set, tagging a backlog. A **batch API** takes a whole file of requests,
runs them **asynchronously** when the provider has spare capacity, and gives the results back later.
In exchange for waiting (up to 24 hours), you pay about half: the **batch discount**. Each request carries
your own key, because results can come back in any order and some may fail on their own.

Analogy: dropping a sack of shirts at the laundry for tomorrow, instead of paying the express counter
to iron them while you wait.

## Picture
```mermaid
flowchart LR
    N["50 notes"] --> L["50 live calls<br/>full price, ~1 s each"]
    N --> F["requests.jsonl<br/>key + request per line"]
    F --> J["batch job<br/>PENDING -> RUNNING -> SUCCEEDED"]
    J -->|poll| R["results by key<br/>any order, some errors"]
    L --> C{"compare<br/>$ and wait"}
    R --> C
```

## Step by step
### 1. Start the recorder
Live calls are recorded once to `../data/llm_cassettes/42_batch-processing.yaml` and replayed after that.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("42_batch-processing")

LLM mode: replay (42_batch-processing.yaml)


### 2. Setup from lessons 07 and 12
The client, the pinned model, the price table, `cost_usd` and `usage_tokens`, unchanged. New: 50 longer
research notes in `../data/notes_50.json`, so there is something worth summarising.

In [2]:
import os, json, pathlib, time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
PRICES_PER_MTOK = {"gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50}}   # Gemini pricing page, 2026-10-06

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

def usage_tokens(resp) -> tuple[int, int]:
    u = resp.usage_metadata
    return u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)

NOTES_50 = json.loads(pathlib.Path("../data/notes_50.json").read_text())
print(len(NOTES_50), "notes; model:", MODEL, "| note 1:", NOTES_50[0]["text"][:60], "...")

50 notes; model: gemini-3.5-flash-lite | note 1: Mars has two small moons, Phobos and Deimos. Phobos orbits s ...


### 3. One live summary, timed
The summarising rules live in the system instruction. `summarise_live` times only the call itself: while
recording, the course spaces calls 4 s apart, so it waits that out first and starts the stopwatch after.

In [3]:
SUMMARY_RULES = "Summarise the research note in one line of at most 12 words. Reply with the line only."
SUMMARY_CONFIG = types.GenerateContentConfig(system_instruction=SUMMARY_RULES, max_output_tokens=40,
                                             temperature=0, automatic_function_calling=NO_AFC)

def summarise_live(note: dict) -> dict:
    if llm_replay.mode() in ("record", "live"):        # skip the course's call spacing, so we time the call only
        time.sleep(float(os.environ.get("TUT_LLM_MIN_INTERVAL", "4")))
    start = time.perf_counter()
    resp = client.models.generate_content(model=MODEL, contents=note["text"], config=SUMMARY_CONFIG)
    tokens_in, tokens_out = usage_tokens(resp)
    return {"summary": resp.text.strip(), "in": tokens_in, "out": tokens_out,
            "secs": round(time.perf_counter() - start, 3)}

print("ready")

ready


The smallest example: note 1.

In [4]:
one = summarise_live(NOTES_50[0])
print("summary:", one["summary"])
print(f"usage: {one['in']} in, {one['out']} out -> ${cost_usd(MODEL, one['in'], one['out']):.6f}")

summary: Mars's inner moon Phobos is spiralling inward and may form rings.
usage: 68 in, 16 out -> $0.000060


Fractions of a cent, as expected for a short note. Now all 50, one after another.

### 4. Fifty live calls
Replay returns instantly, so timings measured now would be meaningless. On a real run (record or live)
the cell saves the measured seconds to `../data/recorded/42.json`; the next cell always reads them from there.

In [5]:
live = {}
for note in NOTES_50:
    live[note["id"]] = summarise_live(note)
if llm_replay.mode() != "replay":
    pathlib.Path("../data/recorded/42.json").write_text(json.dumps(live, indent=1))
for i in (1, 25, 50):
    print(f"note {i:>2}: {live[i]['summary']}")

note  1: Mars' inner moon Phobos is spiralling inward and may form a ring.
note 25: Launched in 2021, the James Webb Space Telescope observes infrared from L2.
note 50: Launching rockets eastward near the equator provides a free speed boost.


If you skipped the model cells, this loads the recorded run. Either way, the timings come from the real run.

In [6]:
recorded = {int(k): v for k, v in json.loads(pathlib.Path("../data/recorded/42.json").read_text()).items()}
if "live" not in globals():
    live = recorded                                   # recorded output of step 4
secs = sorted(r["secs"] for r in recorded.values())   # recorded timings of the real run
live_in, live_out = sum(r["in"] for r in live.values()), sum(r["out"] for r in live.values())
live_cost = cost_usd(MODEL, live_in, live_out)
print(f"50 live calls: {live_in} in + {live_out} out tokens = ${live_cost:.5f}")
print(f"per call: median {secs[25]:.2f} s, slowest {secs[-1]:.2f} s; one after another: {sum(secs):.0f} s")

50 live calls: 2791 in + 823 out tokens = $0.00289
per call: median 0.58 s, slowest 3.54 s; one after another: 34 s


The median call took about half a second and even the slowest only a few seconds; the whole run was
well under a minute of model time. That speed is what you pay full price for.

### 5. The same work as a batch file
A batch job takes a JSONL file: one JSON object per line, each with your **key** and a normal
`generateContent` request. This is the Gemini API's documented format. The key is how you match results.

In [7]:
def batch_line(note: dict) -> dict:
    return {"key": f"note-{note['id']}",
            "request": {"contents": [{"role": "user", "parts": [{"text": note["text"]}]}],
                        "systemInstruction": {"parts": [{"text": SUMMARY_RULES}]},
                        "generationConfig": {"maxOutputTokens": 40, "temperature": 0}}}

LINES = [batch_line(n) for n in NOTES_50]
REQUESTS_PATH = pathlib.Path("../data/batch_42/requests.jsonl")
REQUESTS_PATH.parent.mkdir(parents=True, exist_ok=True)
REQUESTS_PATH.write_text("\n".join(json.dumps(line) for line in LINES) + "\n")
print(REQUESTS_PATH.stat().st_size, "bytes,", len(LINES), "lines; first key:", LINES[0]["key"])

21546 bytes, 50 lines; first key: note-1


About 20 KB. The Gemini API accepts up to 20 MB inline or a 2 GB file; Anthropic up to 100,000 requests
or 256 MB per batch; Vertex up to 200,000 requests from Cloud Storage or BigQuery.

### 6. Submitting to a real batch API
What happened on this course's machine: Vertex express mode (our API key) rejects inline requests
(see Common mistake 1) and needs a Cloud Storage bucket, which an API key alone does not give you. So this cell uses the Gemini Developer API and needs a `GEMINI_API_KEY`. It was **not run** while
building the lesson. It is tagged `needs-cloud`, and the next cell falls back to a local stand-in.

In [8]:
gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])           # Gemini Developer API, not Vertex
uploaded = gclient.files.upload(file=str(REQUESTS_PATH),
                                config=types.UploadFileConfig(display_name="scout-42", mime_type="jsonl"))
cloud_job = gclient.batches.create(model=MODEL, src=uploaded.name, config={"display_name": "scout-summaries-42"})
print(cloud_job.name, cloud_job.state.name)       # later: gclient.batches.get(name=cloud_job.name)

⏭  Skipped: this cell needs [cloud]. Run it yourself, or re-run with TUT_ENABLE=cloud.


### 7. A local stand-in with the same lifecycle
`LocalBatchService` copies the shape of a real batch API: create returns a job name, the job moves
PENDING -> RUNNING -> SUCCEEDED on a **simulated clock**, results come back shuffled, and a bad request fails
alone. It has no model: it "summarises" by keeping the first 12 words. Its timings are invented settings,
not measurements.

In [9]:
import random

class LocalBatchService:
    QUEUE_MIN, RUN_MIN = 20, 40          # simulated minutes; real jobs: anything up to the 24 h target

    def __init__(self, runner, seed: int = 42):
        self.runner, self.rng, self.jobs = runner, random.Random(seed), {}

    def create(self, lines: list[dict]) -> str:
        name = f"batches/local-{len(self.jobs) + 1}"
        self.jobs[name] = {"lines": lines, "clock": 0, "results": None}
        return name

    def get(self, name: str, minutes_waited: int) -> str:
        job = self.jobs[name]
        job["clock"] += minutes_waited
        if job["clock"] < self.QUEUE_MIN:
            return "JOB_STATE_PENDING"
        if job["clock"] < self.QUEUE_MIN + self.RUN_MIN:
            return "JOB_STATE_RUNNING"
        if job["results"] is None:
            job["results"] = [self._run(line) for line in job["lines"]]
            self.rng.shuffle(job["results"])                   # real APIs do not keep input order either
        return "JOB_STATE_SUCCEEDED"

    def _run(self, line: dict) -> dict:
        try:
            return {"key": line["key"], "response": self.runner(line["request"])}
        except ValueError as e:
            return {"key": line["key"], "error": str(e)}

The runner is a stand-in for the model. An empty note raises, like Gemini's "Model input cannot be empty"
(lesson 08), so we can see a partial failure.

In [10]:
def first_words(request: dict) -> str:
    text = request["contents"][0]["parts"][0]["text"].strip()
    if not text:
        raise ValueError("400 INVALID_ARGUMENT: model input cannot be empty")
    return " ".join(text.split()[:12])

bad_line = batch_line({"id": 51, "text": "   "})     # a note that was saved empty by mistake
print(first_words(LINES[0]["request"]))

Mars has two small moons, Phobos and Deimos. Phobos orbits so close


### 8. Submit, poll, collect by key
Poll with a growing wait (5, 10, 20, up to 60 minutes) instead of hammering the status endpoint.
Here the clock is simulated, so nothing actually sleeps.

In [11]:
service = LocalBatchService(first_words)
job_name = service.create(LINES + [bad_line])
waited, delay, state = 0, 5, "JOB_STATE_PENDING"
while state not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_EXPIRED", "JOB_STATE_CANCELLED"):
    state = service.get(job_name, delay)
    waited += delay
    print(f"after {waited:>3} simulated min: {state}")
    delay = min(delay * 2, 60)

after   5 simulated min: JOB_STATE_PENDING
after  15 simulated min: JOB_STATE_PENDING
after  35 simulated min: JOB_STATE_RUNNING
after  75 simulated min: JOB_STATE_SUCCEEDED


Five polls instead of one per minute. Now read the results. Never trust their order: index them by key.

In [12]:
results = service.jobs[job_name]["results"]
print("first three keys returned:", [r["key"] for r in results[:3]])
by_key = {r["key"]: r for r in results}
ok = {k: r["response"] for k, r in by_key.items() if "response" in r}
failed = {k: r["error"] for k, r in by_key.items() if "error" in r}
print(f"{len(ok)} succeeded, {len(failed)} failed: {failed}")
print("note-25:", ok["note-25"])

first three keys returned: ['note-26', 'note-24', 'note-20']
50 succeeded, 1 failed: {'note-51': '400 INVALID_ARGUMENT: model input cannot be empty'}
note-25: The James Webb Space Telescope launched in December 2021. It observes in


The results are shuffled, but `by_key` puts each answer back with its note. One request failed and the
other 50 were fine. On Anthropic's batch API, errored requests are not billed. Check your provider. Fix the note and resubmit only that key.

### 9. Compare cost and wait
The batch bills the same tokens at the documented 50% discount (Gemini, Vertex, Anthropic and OpenAI all
list −50%). The token counts are the real ones from step 4.

In [13]:
BATCH_DISCOUNT = 0.5          # documented by all three vendors (RESEARCH.md); check your provider's page
batch_cost = live_cost * BATCH_DISCOUNT
print(f"50 notes   live ${live_cost:.5f}  batch ${batch_cost:.5f}")
for per_day in (5_000, 500_000):
    scale = per_day / 50 * 30
    print(f"{per_day:>7,}/day: live ${live_cost * scale:,.2f}/month, batch ${batch_cost * scale:,.2f}/month")
print(f"wait: live {sum(secs):.0f} s of model time; batch: target up to 24 h (Gemini API), "
      "most within 1 h (Anthropic)")

50 notes   live $0.00289  batch $0.00145
  5,000/day: live $8.68/month, batch $4.34/month
500,000/day: live $868.44/month, batch $434.22/month
wait: live 34 s of model time; batch: target up to 24 h (Gemini API), most within 1 h (Anthropic)


For 50 notes the saving is far below a cent; nobody should wait hours for that. At half a million notes a
day it is real money. The price of the discount is the wait, so the question is who is waiting.

### 10. When is the wait acceptable?
A small rule Scout's team can apply to each job. `DOCUMENTED_MAX_H` is how long a job can take before
it expires: Anthropic 24 h, Gemini API 48 h; Vertex may queue up to 72 h.

In [14]:
DOCUMENTED_MAX_H = {"anthropic": 24, "gemini-api": 48, "vertex": 72}

def choose_mode(someone_waiting: bool, deadline_hours: float, provider: str, monthly_saving_usd: float) -> str:
    if someone_waiting:
        return "live: a person is waiting for the answer"
    if deadline_hours < DOCUMENTED_MAX_H[provider]:
        return "live (or batch + live fallback): batch may not finish before the deadline"
    if monthly_saving_usd < 10:
        return "live: saving too small to justify a second code path"
    return "batch"

monthly = (live_cost - batch_cost) * 500_000 / 50 * 30
print("user asks Scout a question :", choose_mode(True, 0.01, "gemini-api", monthly))
print("nightly re-summary, due 9am:", choose_mode(False, 12, "gemini-api", monthly))
print("weekly backfill of 500k/day:", choose_mode(False, 7 * 24, "gemini-api", monthly))

user asks Scout a question : live: a person is waiting for the answer
nightly re-summary, due 9am: live (or batch + live fallback): batch may not finish before the deadline
weekly backfill of 500k/day: batch


Chat stays live. A nightly job due in 12 hours cannot rely on a 48-hour limit. A weekly backfill with a
real saving goes to batch.

We compare against the expiry limit, not the 24 h target, because an expired job returns nothing. Add
your own safety margin.

## At work
- **Offline work goes to batch:** nightly re-summaries, classification backlogs and eval runs, where nobody
  waits and the 50% discount adds up. → lesson 48
- **Discounts combine differently by vendor.** Anthropic's prompt cache (a cheaper price for re-reading a
  repeated prompt prefix) and batch discounts stack. On Vertex they do not: cached tokens get the cache
  discount instead.
  Model the real bill per vendor. → lesson 54

- **What breaks:** results matched by position, jobs that expire (24 h Anthropic, 48 h Gemini API) and
  return nothing, and full resubmissions that pay twice. Teams store each key's state, resubmit only
  missing keys, and log job ids. → lessons 39 and 49
- **Spend caps lag behind batches.** Google warns that batch jobs can push spend past a project's monthly
  cap, because billing data arrives late. Put a hard limit on batch size in your own code too. → lesson 54

## What just happened
- One live summary cost a fraction of a cent; 50 live calls took about half a second each (median, recorded).
- The same 50 requests became `requests.jsonl`: one `{"key", "request"}` line each, about 20 KB.
- Vertex express mode could not run the batch (no inline requests, needs a bucket); the real Gemini API
  code is shown under `needs-cloud`, and a local stand-in ran the same lifecycle on a simulated clock.
- Results came back shuffled with one error; indexing by key matched all 50 good answers correctly.
- Batch halves the bill; `choose_mode` keeps anything a person waits for, or anything due before the
  provider's time limit, live.

## Common mistakes
**1. Sending inline requests to Vertex.** Inline batches are a Gemini Developer API feature. The Vertex
client raises a ValueError.

In [15]:
client.batches.create(model=MODEL, src=[types.InlinedRequest(contents=NOTES_50[0]["text"], model=MODEL)])

ValueError: Exactly one of `gcs_uri` or `bigquery_uri`, or `vertex_dataset_name` must be set, other sources are not supported in Gemini Enterprise Agent Platform.

Fix: on Vertex, point `src` at a JSONL file in Cloud Storage (`gs://...`) or a BigQuery table and set a
`dest`; inline lists work only with the Gemini Developer API client.

**2. Matching results by position.** Zipping results with the notes in input order quietly attaches
summaries to the wrong notes. A check exposes it:

In [16]:
for note, result in zip(NOTES_50, results):
    assert result["key"] == f"note-{note['id']}", f"note {note['id']} got {result['key']}'s result"

AssertionError: note 1 got note-26's result

Fix: build `by_key = {r["key"]: r for r in results}` and look each note up by its key, as in step 8.

## Try it
No new model calls: reuse `live`, `service`, `first_words` and `choose_mode`.
1. Fix the empty note (give it any text), submit only that key as a new batch on `service`, poll it to
   the end, and merge its answer into `ok`, so you have 51 summaries.
2. A team needs 2,000 summaries within 30 hours, on Vertex. What does `choose_mode` say, and what is
   the monthly saving if they run it daily anyway?

In [17]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [18]:
llm_replay.stop()

## Key terms
- **batch API** — an endpoint that accepts many model requests at once (often a JSONL file), runs them later, and returns all results together, at a discount.
- **asynchronous job** — work you submit and check on later by polling its state, instead of waiting for the reply in the same call.
- **batch discount** — the lower price for batched requests; 50% of the standard input and output price at Gemini, Vertex, Anthropic and OpenAI.
- **request key (custom_id)** — your own id on each batched request, used to match results, which can come back in any order.
- **job state** — where a batch job is in its lifecycle, e.g. Gemini's `JOB_STATE_PENDING`, `RUNNING`, `SUCCEEDED`, `FAILED`, `EXPIRED`.
- **turnaround time** — how long a batch job takes from submission to results; providers document targets (24 h) and expiry limits, not guarantees.